# 01 — The ewm-sm sidecar surface (`ewm-scene`)

**Question this notebook answers:** what exactly crosses the boundary
between the Python controller and the HLLSet lattice?

Every ewm-sm interaction is a subprocess call: a JSONL file goes in, one
JSON document comes out. This notebook runs each flat-frame command and
shows the raw output. Companion reference: `docs/INTERACTION_MAP.md` §1.


In [1]:
import os, sys

# make the `lab` package importable whether jupyter starts here or in the repo root
for _p in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.exists(os.path.join(_p, "lab")):
        sys.path.insert(0, _p)
        PROJ = _p
        break
else:
    raise RuntimeError("run jupyter from the project root or the notebooks/ directory")

from lab import EwmScene, LayaDaemon, BonsaiClient
print("project root:", PROJ)



project root: /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-laya-bonsai-lab


In [2]:
ewm = EwmScene()
print("resolved binary:", ewm.bin)


resolved binary: /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/target/release/ewm-scene


## Frames in

The controller's lingua franca is `tid{n}` strings — Bonsai token ids
lowered to strings. Here is a three-frame "conversation": frame 2 keeps
`tid2`/`tid3`, drops `tid1`/`tid4`, and adds `tid5`.


In [3]:
import json, os

WORK = os.path.join(PROJ, "work", "nb01")
os.makedirs(WORK, exist_ok=True)
union = os.path.join(WORK, "frames.jsonl")

frames = [
    {"id": 1, "tokens": ["tid1", "tid2", "tid3", "tid4"]},
    {"id": 2, "tokens": ["tid2", "tid3", "tid5"]},
    {"id": 3, "tokens": ["tid5", "tid6", "tid7"]},
]
with open(union, "w") as fh:
    for f in frames:
        fh.write(json.dumps(f) + "\n")
print(open(union).read())


{"id": 1, "tokens": ["tid1", "tid2", "tid3", "tid4"]}
{"id": 2, "tokens": ["tid2", "tid3", "tid5"]}
{"id": 3, "tokens": ["tid5", "tid6", "tid7"]}



## `ingest` — per-frame content key and popcount

`pop` is the HLLSet popcount — roughly `3 × distinct tokens` because each
token is ingested with three seeds. `key` is the content address of the
frame (`h:<sha1…>`).


In [4]:
ewm.ingest(union, verbose=True)


$ /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/target/release/ewm-scene ingest /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-laya-bonsai-lab/work/nb01/frames.jsonl
{
  "frames": [
    {
      "id": 1,
      "key": "h:e0ac0a66ea0ff17c1359a7effe50c7790731294a",
      "pop": 12
    },
    {
      "id": 2,
      "key": "h:4a41f38507fa74d28e1a18d4a062189b38be40e1",
      "pop": 9
    },
    {
      "id": 3,
      "key": "h:0332701d6418b8e1a6097c240d6dcc92f9840b34",
      "pop": 9
    }
  ]
}


{'frames': [{'id': 1,
   'key': 'h:e0ac0a66ea0ff17c1359a7effe50c7790731294a',
   'pop': 12},
  {'id': 2, 'key': 'h:4a41f38507fa74d28e1a18d4a062189b38be40e1', 'pop': 9},
  {'id': 3, 'key': 'h:0332701d6418b8e1a6097c240d6dcc92f9840b34', 'pop': 9}]}

## `bss` — consecutive similarity

`tau[i]` is BSSτ = `|A ∩ B| / |B|` between frames i and i+1 ("how much of
the new frame was already in the old one"). `jaccard[i]` is
`|A ∩ B| / |A ∪ B|`.


In [5]:
ewm.bss(union, verbose=True)


$ /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/target/release/ewm-scene bss /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-laya-bonsai-lab/work/nb01/frames.jsonl
{
  "jaccard": [
    0.4,
    0.2
  ],
  "keys": [
    "h:e0ac0a66ea0ff17c1359a7effe50c7790731294a",
    "h:4a41f38507fa74d28e1a18d4a062189b38be40e1",
    "h:0332701d6418b8e1a6097c240d6dcc92f9840b34"
  ],
  "pop": [
    12,
    9,
    9
  ],
  "tau": [
    0.6666666666666666,
    0.3333333333333333
  ]
}


{'jaccard': [0.4, 0.2],
 'keys': ['h:e0ac0a66ea0ff17c1359a7effe50c7790731294a',
  'h:4a41f38507fa74d28e1a18d4a062189b38be40e1',
  'h:0332701d6418b8e1a6097c240d6dcc92f9840b34'],
 'pop': [12, 9, 9],
 'tau': [0.6666666666666666, 0.3333333333333333]}

## `noether` — the D/R/N decomposition

Per transition: `dp` (departed), `rp` (retained), `np` (new) popcounts,
plus three indicators:

- `ind1 = dp / |R ∪ N|` — departure dominates
- `ind3 = np / |R ∪ D|` — arrival dominates
- `ind2 = BSS(R_prev, R)` — retention-chain stability (a **similarity**,
  so low = structurally surprising; it starts at transition 2)


In [6]:
ewm.noether(union, verbose=True)


$ /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/target/release/ewm-scene noether /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-laya-bonsai-lab/work/nb01/frames.jsonl
{
  "dp": [
    6,
    6
  ],
  "ind1": [
    0.6666666666666666,
    0.6666666666666666
  ],
  "ind2": [
    0.0
  ],
  "ind3": [
    0.25,
    0.6666666666666666
  ],
  "np": [
    3,
    6
  ],
  "rp": [
    6,
    3
  ]
}


{'dp': [6, 6],
 'ind1': [0.6666666666666666, 0.6666666666666666],
 'ind2': [0.0],
 'ind3': [0.25, 0.6666666666666666],
 'np': [3, 6],
 'rp': [6, 3]}

## `materialize` — token restoration

The lattice restores each frame's tokens three ways: `ordered` (greedy
restoration of order), `set` (plain set), `beam2` (beam-2). The controller
uses `ordered` for full memory.

> Edge case: `materialize` panics on frames with 1–2 tids ("attempt to
> subtract with overflow"). `ingest`/`noether` are safe on tiny frames.


In [7]:
ewm.materialize(union, verbose=True)


$ /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/target/release/ewm-scene materialize /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-laya-bonsai-lab/work/nb01/frames.jsonl --beam 2
{
  "frames": [
    {
      "beam2": [
        "tid1",
        "tid2",
        "tid3",
        "tid4"
      ],
      "id": 1,
      "ordered": [
        "tid1",
        "tid2",
        "tid3",
        "tid4"
      ],
      "set": [
        "tid1",
        "tid2",
        "tid3",
        "tid4"
      ]
    },
    {
      "beam2": [
        "tid2",
        "tid3",
        "tid5"
      ],
      "id": 2,
      "ordered": [
        "tid2",
        "tid3",
        "tid5"
      ],
      "set": [
        "tid2",
        "tid3",
        "tid5"
      ]
    },
    {
      "beam2": [
        "tid5",
        "tid6",
        "tid7"
      ],
      "id": 3,
      "ordered": [
        "tid5",
        "tid6",
        "tid7"
      ],
      "set": [
        "tid5",
        "tid6",
        "tid7"
      ]
  

{'frames': [{'beam2': ['tid1', 'tid2', 'tid3', 'tid4'],
   'id': 1,
   'ordered': ['tid1', 'tid2', 'tid3', 'tid4'],
   'set': ['tid1', 'tid2', 'tid3', 'tid4']},
  {'beam2': ['tid2', 'tid3', 'tid5'],
   'id': 2,
   'ordered': ['tid2', 'tid3', 'tid5'],
   'set': ['tid2', 'tid3', 'tid5']},
  {'beam2': ['tid5', 'tid6', 'tid7'],
   'id': 3,
   'ordered': ['tid5', 'tid6', 'tid7'],
   'set': ['tid5', 'tid6', 'tid7']}]}

## `sidecar` — the Boolean-ring trajectory

Each frame is measured against the growing GF(2) basis *before* it is
inserted: `soft` (BSS weights over basis elements), `hard` (GF(2)
coordinates when in-span), `residual` (linear novelty), `step` (soft-key
L2 distance from the previous frame), and a jump detector over the step
series (`jumps`, threshold = mean + 3σ). `--freeze N` stops basis updates
after N frames so later soft keys share one coordinate system.


In [8]:
ewm.sidecar(union, freeze=3, verbose=True)


$ /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/target/release/ewm-scene sidecar /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-laya-bonsai-lab/work/nb01/frames.jsonl --freeze 3
{
  "basis_history": [
    {
      "cover_pop": 12,
      "dimension": 1,
      "generation": 1
    },
    {
      "cover_pop": 21,
      "dimension": 2,
      "generation": 2
    },
    {
      "cover_pop": 33,
      "dimension": 3,
      "generation": 3
    }
  ],
  "frames": [
    {
      "basis_pop": [],
      "dim": 1,
      "hard": null,
      "id": 1,
      "in_span": false,
      "residual": 12,
      "rotation_count": 0,
      "rotation_mass": 0,
      "soft": [],
      "soft_first": [
        1.0
      ],
      "spill_first": 0,
      "step": 0.0
    },
    {
      "basis_pop": [
        12
      ],
      "dim": 2,
      "hard": null,
      "id": 2,
      "in_span": false,
      "residual": 9,
      "rotation_count": 0,
      "rotation_mass": 0,
      "soft": [
        0.5
     

{'basis_history': [{'cover_pop': 12, 'dimension': 1, 'generation': 1},
  {'cover_pop': 21, 'dimension': 2, 'generation': 2},
  {'cover_pop': 33, 'dimension': 3, 'generation': 3}],
 'frames': [{'basis_pop': [],
   'dim': 1,
   'hard': None,
   'id': 1,
   'in_span': False,
   'residual': 12,
   'rotation_count': 0,
   'rotation_mass': 0,
   'soft': [],
   'soft_first': [1.0],
   'spill_first': 0,
   'step': 0.0},
  {'basis_pop': [12],
   'dim': 2,
   'hard': None,
   'id': 2,
   'in_span': False,
   'residual': 9,
   'rotation_count': 0,
   'rotation_mass': 0,
   'soft': [0.5],
   'soft_first': [0.5],
   'spill_first': 3,
   'step': 0.5},
  {'basis_pop': [12, 9],
   'dim': 3,
   'hard': None,
   'id': 3,
   'in_span': False,
   'residual': 12,
   'rotation_count': 2,
   'rotation_mass': 24,
   'soft': [0.0, 0.3333333333333333],
   'soft_first': [0.0],
   'spill_first': 9,
   'step': 0.8333333333333334}],
 'freeze': 3,
 'jumps': [],
 'threshold': 1.1666666666666667}

## `ma` — moving-average BSS over trailing windows

`fast`/`slow` are BSS between consecutive trailing-union windows of length
`--short` / `--long`. They are the cheapest "is the conversation changing
pace?" signal.


In [9]:
ewm.ma(union, short=1, long=2, verbose=True)


$ /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/target/release/ewm-scene ma /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-laya-bonsai-lab/work/nb01/frames.jsonl --short 1 --long 2
{
  "fast": [
    0.3333333333333333
  ],
  "slow": [
    0.6
  ],
  "t0": [
    3
  ]
}


{'fast': [0.3333333333333333], 'slow': [0.6], 't0': [3]}